# Solution PromoPro — évaluation du modèle final sur un plan

À lancer **après** le carnet d'entraînement, cellule par cellule (il demande deux fichiers). Il lit `promopro-plan3d.onnx` sur votre Drive (ou le demande), segmente `villa_plan_2d.png` en letterbox comme à l'entraînement, extrait les pièces (murs, portes et fenêtres font barrière, ouvertures de porte refermées), puis compare au JSON de référence Gemini (`villa_plan_2d.gemini.json`, 9 pièces). Pas de GPU nécessaire.

In [ ]:
# 1. Dépendances et modèle ONNX
%pip -q install onnxruntime opencv-python-headless 'protobuf<6'
import os, json, numpy as np, cv2, onnxruntime as ort
from google.colab import files
ONNX = '/content/drive/MyDrive/promopro-ia/promopro-plan3d.onnx'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive non monté :', e)
if not os.path.exists(ONNX):
    print('Déposez promopro-plan3d.onnx :'); ONNX = next(iter(files.upload()))
sess = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])
TAILLE = sess.get_inputs()[0].shape[-1] if isinstance(sess.get_inputs()[0].shape[-1], int) else 512
print('modèle chargé :', ONNX, '| taille d\'entrée', TAILLE)

In [ ]:
# 2. Déposez l'image du plan (villa_plan_2d.png) — inférence en letterbox
chemin = next(iter(files.upload()))
img = cv2.cvtColor(cv2.imread(chemin), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
ech = TAILLE / max(H, W); redim = cv2.resize(img, (round(W * ech), round(H * ech)), interpolation=cv2.INTER_AREA)
toile = np.full((TAILLE, TAILLE, 3), 255, np.uint8); toile[:redim.shape[0], :redim.shape[1]] = redim
moy, ecart = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
x = ((toile.astype(np.float32) / 255 - moy) / ecart).transpose(2, 0, 1)[None].astype(np.float32)
logits = sess.run(None, {'image': x})[0]
masque = cv2.resize(logits[0].argmax(0).astype(np.uint8)[:redim.shape[0], :redim.shape[1]], (W, H), interpolation=cv2.INTER_NEAREST)
print('pixels par classe (0 fond, 1 mur, 2 porte, 3 fenêtre) :', {c: int((masque == c).sum()) for c in range(4)})
couleurs = np.array([[255, 255, 255], [20, 20, 20], [230, 120, 0], [0, 120, 230]], dtype=np.uint8)
cv2.imwrite('masque.png', cv2.cvtColor((0.55 * img + 0.45 * couleurs[masque]).astype(np.uint8), cv2.COLOR_RGB2BGR))
cv2.imwrite('masque_seul.png', cv2.cvtColor(couleurs[masque], cv2.COLOR_RGB2BGR))
from IPython.display import Image, display
display(Image('masque.png', width=700)); display(Image('masque_seul.png', width=700))

In [ ]:
# 3. Pièces et portes (post-traitement : barrière = murs + portes + fenêtres, ouvertures refermées)
def extraire(masque, fermeture_px):
    murs = (masque == 1).astype(np.uint8); barriere = (masque != 0).astype(np.uint8)
    if fermeture_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (fermeture_px, fermeture_px))
        barriere = np.maximum(barriere, cv2.morphologyEx(murs, cv2.MORPH_CLOSE, k))
    barriere = cv2.dilate(barriere, np.ones((7, 7), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats((1 - barriere).astype(np.uint8), connectivity=4)
    pieces = []
    for i in range(1, n):
        x0, y0, w, h, aire = stats[i]
        if aire < 0.004 * H * W or x0 == 0 or y0 == 0 or x0 + w == W or y0 + h == H: continue
        pieces.append({'nom': f'Pièce {len(pieces) + 1}', 'x': round(x0 / W, 3), 'y': round(y0 / H, 3), 'largeur': round(w / W, 3), 'hauteur': round(h / H, 3)})
    return pieces
porte = (masque == 2).astype(np.uint8)
n2, _, stats2, centres = cv2.connectedComponentsWithStats(porte, connectivity=8)
portes = []
for i in range(1, n2):
    x0, y0, w, h, aire = stats2[i]
    if aire < 0.0002 * H * W or max(w, h) > 0.15 * max(H, W): continue
    portes.append({'x': round(centres[i][0] / W, 3), 'y': round(centres[i][1] / H, 3), 'mur': 'vertical' if h > w else 'horizontal'})
FERMETURE = int(0.04 * max(H, W))  # ~ une largeur de porte
resultats = {'sans fermeture': extraire(masque, 0), f'fermeture {FERMETURE} px': extraire(masque, FERMETURE)}
for nom, p in resultats.items(): print(f'{nom} : {len(p)} pièce(s)')
print(f'portes détectées : {len(portes)}')
json.dump({'pieces': resultats[f'fermeture {FERMETURE} px'], 'portes': portes, 'remarques': 'modèle Solution PromoPro (ONNX)'}, open('villa_plan_2d.promopro.json', 'w'), ensure_ascii=False, indent=2)
files.download('villa_plan_2d.promopro.json'); files.download('masque_seul.png')

In [ ]:
# 4. Déposez villa_plan_2d.gemini.json (référence, 9 pièces) — comparaison IoU par pièce
gem = json.load(open(next(iter(files.upload()))))
assert 'modèle' not in str(gem.get('remarques', '')), 'Ce fichier vient d\'un modèle : déposez villa_plan_2d.gemini.json.'
def iou(a, b):
    ax2, ay2, bx2, by2 = a['x'] + a['largeur'], a['y'] + a['hauteur'], b['x'] + b['largeur'], b['y'] + b['hauteur']
    inter = max(0, min(ax2, bx2) - max(a['x'], b['x'])) * max(0, min(ay2, by2) - max(a['y'], b['y']))
    union = a['largeur'] * a['hauteur'] + b['largeur'] * b['hauteur'] - inter
    return inter / union if union else 0
for nom, pieces in resultats.items():
    print(f'\n=== {nom} ({len(pieces)} pièces du modèle)')
    print(f"{'pièce Gemini':<18}{'meilleure pièce modèle':<24}{'IoU':>6}")
    ok = 0; usage = {}
    for p in gem['pieces']:
        meilleur = max(pieces, key=lambda q: iou(p, q), default=None); s = iou(p, meilleur) if meilleur else 0; ok += s >= 0.7
        if meilleur: usage[meilleur['nom']] = usage.get(meilleur['nom'], 0) + 1
        print(f"{p['nom']:<18}{(meilleur['nom'] if meilleur else '—'):<24}{s:>6.2f}")
    fusions = [f'{k} couvre {v} pièces' for k, v in usage.items() if v > 1]
    print(f"pièces retrouvées (IoU ≥ 0,7) : {ok}/{len(gem['pieces'])}" + (' — fusions : ' + ' ; '.join(fusions) if fusions else ''))
print(f"\nportes : Gemini {len(gem['portes'])}, modèle {len(portes)}")